# AIC 2026 — Vòng Sơ tuyển (bộ đề Thử nghiệm P1) — Pipeline truy vấn video

Notebook chạy trên **Kaggle 2×GPU 16GB**, sinh ra `submission.zip` cho 3 loại truy vấn:
**Textual KIS**, **Q&A**, **TRAKE**.

**Kiến trúc**

```
query .txt (tiếng Việt)
   │
   ├─(1) Query Understanding  ── LLM API (claude-opus-5) ──► topic_en, clip_prompts[], keywords_vi/en,
   │                                                          ocr_terms[], objects[], events[]
   ├─(2) Video-level retrieval  (BM25 + dense trên Summary + Transcript(vi/en) + media-info) ──► video prior
   │
   ├─(3) Keyframe-level retrieval (177k keyframes)
   │        • CLIP ViT-B/32  (feature BTC cấp)  × text prompt
   │        • Dense multilingual-e5  trên  caption(Florence-2) + OCR
   │        • BM25  trên  caption + OCR + ASR-window (vi & en)
   │        • OCR exact-match bonus + Object (Faster R-CNN / OpenImages) bonus
   │
   ├─(4) Fusion (weighted normalized scores + video prior)  ──► candidate pool
   ├─(5) Rerank  ── LLM (text evidence) và/hoặc VLM (ảnh keyframe, tuỳ chọn)
   └─(6) Sinh 100 dòng/CSV theo từng loại truy vấn (+ hedging frame-spread)
```

**Điểm quan trọng về cách tính điểm** — `Final = mean(R@1, R@5, R@20, R@50, R@100)`.
Notebook vì vậy không chỉ xếp hạng mà còn **hedge**: dòng đầu là ứng viên tốt nhất, các dòng
kế tiếp trải frame trong cùng shot và các shot lân cận để tăng khả năng có một dòng rơi vào
khoảng `[s, e]` của đáp án (khoảng này rất hẹp — KIS ~10 frame, TRAKE < 10 frame).

**Chuẩn bị dữ liệu trên Kaggle** (tên thư mục được tự nhận diện):
- `Feature_Dataset` — clip-features-32, map-keyframes, Image_captioning, OCR_EasyOCR_VietOCR,
  Summary_video, Transcript_Translated, objects-aic25-b1, media-info
- `THUNGHIEM-bo-de-thi` — các file `query-p1-*.txt`
- *(tuỳ chọn)* `Keyframes_L*` → bật `USE_VLM_RERANK`
- *(tuỳ chọn)* `Videos_L*` → bật `USE_VIDEO_FINE_ALIGN` (**rất nên bật cho TRAKE**)

**API key**: Kaggle → Add-ons → Secrets → thêm `ANTHROPIC_API_KEY`.
Không có key thì notebook vẫn chạy ở chế độ fallback (dùng trực tiếp text truy vấn, điểm thấp hơn).

## 1. Config

In [ ]:
# ============================== CONFIG ==============================
CFG = dict(
    # ---- LLM / VLM API ----
    LLM_MODEL          = "claude-opus-5",     # query understanding + rerank + QA
    VLM_MODEL          = "claude-opus-5",     # rerank bằng ảnh keyframe
    USE_LLM            = True,
    USE_LLM_RERANK     = True,   # rerank top-K bằng text evidence
    USE_VLM_RERANK     = False,  # cần dataset Keyframes_L* (bật nếu đã add)
    LLM_RERANK_TOPK    = 40,
    VLM_RERANK_TOPK    = 16,

    # ---- Dense text encoder (GPU) ----
    DENSE_MODEL        = "intfloat/multilingual-e5-large",
    USE_DENSE          = True,
    DENSE_BATCH        = 256,

    # ---- CLIP text encoder (phải khớp feature BTC: clip-ViT-B-32) ----
    CLIP_MODEL         = "clip-ViT-B-32",

    # ---- Features phụ ----
    USE_OBJECTS        = True,   # index ~177k json (lần đầu ~5-10 phút, có cache)
    OBJ_MIN_SCORE      = 0.30,

    # ---- TRAKE fine alignment trên video gốc ----
    USE_VIDEO_FINE_ALIGN = False,   # bật nếu đã add dataset Videos_L*
    FINE_WIN_SEC       = 6.0,       # cửa sổ quét quanh keyframe dự đoán (giây, mỗi bên)
    FINE_STRIDE        = 2,         # bước frame khi quét

    # ---- Fusion weights ----
    W_CLIP     = 1.00,
    W_DENSE    = 0.85,
    W_BM25     = 0.55,
    W_OCR      = 0.45,
    W_OBJ      = 0.25,
    W_VPRIOR   = 0.80,
    POOL_PER_LIST = 3000,

    # ---- Output ----
    N_ROWS     = 100,
    OUT_DIR    = "/kaggle/working/submission",
    ZIP_NAME   = "/kaggle/working/submission.zip",
    CACHE_DIR  = "/kaggle/working/cache",
)
# ====================================================================
import os, json
os.makedirs(CFG["CACHE_DIR"], exist_ok=True)
os.makedirs(CFG["OUT_DIR"], exist_ok=True)
print(json.dumps(CFG, indent=1, ensure_ascii=False))

In [ ]:
!pip -q install bm25s PyStemmer sentence-transformers anthropic

In [ ]:
import os, re, io, json, glob, math, time, zipfile, unicodedata, base64, random
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd, torch

NGPU = torch.cuda.device_count()
DEV  = "cuda:0" if NGPU else "cpu"
print("torch", torch.__version__, "| GPUs:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])

## 2. Tự nhận diện đường dẫn dữ liệu

In [ ]:
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", ".", ".."]

def scan_dirs(max_depth=6):
    """{tên thư mục: [đường dẫn]} — walk có cắt độ sâu và bỏ nhánh L**_V*** để không quét 177k file."""
    found = defaultdict(list)
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        base = root.rstrip("/").count("/")
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count("/") - base >= max_depth:
                dirnames[:] = []
                continue
            dirnames[:] = [d for d in dirnames if not re.fullmatch(r"L\d\d_V\d\d\d", d)]
            found[os.path.basename(dirpath)].append(dirpath)
    return found

DIRS = scan_dirs()

def pick(*names, contains=None, required=True, label=""):
    for n in names:
        for p in DIRS.get(n, []):
            if contains is None or glob.glob(os.path.join(p, contains)):
                return p
    if required:
        raise FileNotFoundError(f"Không tìm thấy {label or names}. Hãy add dataset tương ứng vào notebook.")
    return None

P = {}
P["clip"]  = pick("clip-features-32", contains="*.npy",  label="clip-features-32")
P["map"]   = pick("map-keyframes",    contains="*.csv",  label="map-keyframes")
P["cap"]   = pick("Image_captioning", contains="*.json", required=False)
P["ocr"]   = pick("OCR_EasyOCR_VietOCR", contains="*.json", required=False)
P["summ"]  = pick("Summary_video",    contains="*.json", required=False)
P["media"] = pick("media-info",       contains="*.json", required=False)
P["obj"]   = pick("objects",          required=False)

P["trans"] = None
for cand in DIRS.get("Transcript_Translated", []) + DIRS.get("Transcript_Extract", []):
    if glob.glob(os.path.join(cand, "*", "video", "*.json")):
        P["trans"] = cand
        break

P["query"] = pick("THUNGHIEM-bo-de-thi", contains="query-*.txt", required=False)
if P["query"] is None:
    hits = glob.glob("/kaggle/input/**/query-*-kis.txt", recursive=True)
    P["query"] = os.path.dirname(hits[0]) if hits else None

P["keyframes"] = [p for p in DIRS.get("keyframes", []) if glob.glob(os.path.join(p, "L*_V*"))]
P["videos"]    = sorted({os.path.dirname(p) if os.path.basename(p) == "video" else p
                         for p in glob.glob("/kaggle/input/**/Videos_L*", recursive=True)})

for k, v in P.items():
    print(f"{k:10s} -> {v if not isinstance(v, list) else v[:3]}")
assert P["query"], "Không tìm thấy thư mục chứa query-*.txt"

## 3. Index keyframe + CLIP features

In [ ]:
IDX_CACHE = os.path.join(CFG["CACHE_DIR"], "keyframe_index.parquet")

def build_keyframe_index():
    rows = []
    for csv_path in sorted(glob.glob(os.path.join(P["map"], "*.csv"))):
        df = pd.read_csv(csv_path)
        df["video_id"] = Path(csv_path).stem
        rows.append(df[["video_id", "n", "pts_time", "fps", "frame_idx"]])
    idx = pd.concat(rows, ignore_index=True)
    idx["n"] = idx["n"].astype(np.int32)
    idx["frame_idx"] = idx["frame_idx"].astype(np.int64)
    idx["row"] = np.arange(len(idx), dtype=np.int64)
    return idx

if os.path.exists(IDX_CACHE):
    KF = pd.read_parquet(IDX_CACHE)
else:
    KF = build_keyframe_index()
    KF.to_parquet(IDX_CACHE, index=False)

VIDEOS    = KF["video_id"].drop_duplicates().tolist()
VID2ROWS  = {v: g["row"].to_numpy() for v, g in KF.groupby("video_id", sort=False)}
FRAME_IDX = KF["frame_idx"].to_numpy()
PTS       = KF["pts_time"].to_numpy(np.float32)
FPS_ARR   = KF["fps"].to_numpy(np.float32)
VID_ARR   = KF["video_id"].to_numpy()
KF_N      = KF["n"].to_numpy()
N         = len(KF)
print(f"{N:,} keyframes / {len(VIDEOS)} videos")
KF.head(3)

In [ ]:
CLIP_CACHE = os.path.join(CFG["CACHE_DIR"], "clip_all.npy")

if os.path.exists(CLIP_CACHE):
    CLIP_F = np.load(CLIP_CACHE, mmap_mode="r")
else:
    mats = []
    for v in VIDEOS:
        a = np.load(os.path.join(P["clip"], f"{v}.npy")).astype(np.float32)
        exp = len(VID2ROWS[v])
        if a.shape[0] != exp:                       # an toàn: cắt/pad cho khớp map-keyframes
            print(f"  ! {v}: clip={a.shape[0]} vs map={exp} -> align")
            a = a[:exp] if a.shape[0] > exp else np.vstack(
                [a, np.zeros((exp - a.shape[0], a.shape[1]), np.float32)])
        a /= (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
        mats.append(a.astype(np.float16))
    CLIP_F = np.vstack(mats)
    np.save(CLIP_CACHE, CLIP_F)
    del mats

CLIP_T = torch.from_numpy(np.ascontiguousarray(CLIP_F)).to(DEV)
print("CLIP matrix:", tuple(CLIP_T.shape), CLIP_T.dtype)

## 4. Bằng chứng dạng text: caption / OCR / ASR / summary / media-info

In [ ]:
CAP  = [""] * N          # caption tiếng Anh (Florence-2)
OCRT = [""] * N          # OCR tiếng Việt trên keyframe

row_of = {}              # (video_id, n) -> row
for v, rws in VID2ROWS.items():
    for r_ in rws:
        row_of[(v, int(KF_N[r_]))] = int(r_)

def jload(p):
    try:
        with io.open(p, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

# ---- captions ----
if P["cap"]:
    for v in VIDEOS:
        d = jload(os.path.join(P["cap"], f"{v}.json"))
        if not d:
            continue
        by_n = {}
        for kfd in d.get("keyframes", []):
            n_ = int(kfd.get("n", 0))
            txt = (kfd.get("caption") or "").strip()
            if not txt and kfd.get("duplicate_of"):
                txt = by_n.get(int(kfd["duplicate_of"]), "")
            by_n[n_] = txt
            r = row_of.get((v, n_))
            if r is not None:
                CAP[r] = txt
    print("captions:", sum(1 for x in CAP if x), "/", N)

# ---- OCR (ưu tiên ocr_index.jsonl vì đọc 1 file) ----
ocr_jsonl = next(iter(glob.glob("/kaggle/input/**/ocr_index.jsonl", recursive=True)), None)
if ocr_jsonl:
    with io.open(ocr_jsonl, encoding="utf-8") as f:
        for line in f:
            try:
                d = json.loads(line)
            except Exception:
                continue
            n_ = int(re.sub(r"\D", "", d.get("keyframe", "0")) or 0)
            r = row_of.get((d.get("video_id"), n_))
            if r is not None:
                OCRT[r] = (d.get("text") or "").strip()
elif P["ocr"]:
    for v in VIDEOS:
        d = jload(os.path.join(P["ocr"], f"{v}.json"))
        if not d:
            continue
        for kfd in d.get("keyframes", []):
            r = row_of.get((v, int(kfd.get("n", 0))))
            if r is not None:
                OCRT[r] = (kfd.get("text") or "").strip()
print("ocr:", sum(1 for x in OCRT if x), "/", N)

In [ ]:
# ---- transcript (ASR vi + bản dịch en), summary, media-info ----
TRANS = {}
if P["trans"]:
    for jp in glob.glob(os.path.join(P["trans"], "*", "video", "*.json")):
        d = jload(jp)
        if not d:
            continue
        segs = [(float(s.get("start", 0)), float(s.get("end", 0)),
                 s.get("text") or "", s.get("text_en") or "")
                for s in d.get("segments", [])]
        TRANS[Path(jp).stem] = dict(segments=segs,
                                    vi=d.get("text", "") or "",
                                    en=d.get("text_en", "") or "")
print("transcripts:", len(TRANS))

SUMM = {}
if P["summ"]:
    for v in VIDEOS:
        tp = os.path.join(P["summ"], f"{v}.txt")
        if os.path.exists(tp):
            SUMM[v] = io.open(tp, encoding="utf-8").read().strip()
        else:
            d = jload(os.path.join(P["summ"], f"{v}.json"))
            if d:
                SUMM[v] = (d.get("summary") or "").strip()
print("summaries:", len(SUMM))

MEDIA = {}
if P["media"]:
    for v in VIDEOS:
        d = jload(os.path.join(P["media"], f"{v}.json"))
        if d:
            MEDIA[v] = " ".join(str(d.get(k, "")) for k in
                                ("title", "description", "keywords", "author"))[:4000]
print("media-info:", len(MEDIA))

In [ ]:
# ---- ASR window quanh mỗi keyframe (±6s) ----
ASR_VI = [""] * N
ASR_EN = [""] * N
WIN = 6.0
for v, rws in VID2ROWS.items():
    tr = TRANS.get(v)
    if not tr or not tr["segments"]:
        continue
    segs   = sorted(tr["segments"], key=lambda s: s[0])
    starts = np.array([s[0] for s in segs], np.float32)
    ends   = np.array([s[1] for s in segs], np.float32)
    for r in rws:
        t = PTS[r]
        m = np.where((ends >= t - WIN) & (starts <= t + WIN))[0]
        if len(m):
            ASR_VI[r] = " ".join(segs[i][2] for i in m)[:600]
            ASR_EN[r] = " ".join(segs[i][3] for i in m)[:600]
print("asr windows:", sum(1 for x in ASR_EN if x))

In [ ]:
# ---- Object index (tuỳ chọn, có cache) ----
OBJ_SETS  = [set() for _ in range(N)]
OBJ_CACHE = os.path.join(CFG["CACHE_DIR"], "objects.jsonl")

if CFG["USE_OBJECTS"] and P["obj"]:
    if os.path.exists(OBJ_CACHE):
        with io.open(OBJ_CACHE, encoding="utf-8") as f:
            for line in f:
                r, names = json.loads(line)
                OBJ_SETS[r] = set(names)
    else:
        t0 = time.time()
        with io.open(OBJ_CACHE, "w", encoding="utf-8") as out:
            for vi_, v in enumerate(VIDEOS):
                vdir = os.path.join(P["obj"], v)
                if not os.path.isdir(vdir):
                    continue
                for jp in glob.glob(os.path.join(vdir, "*.json")):
                    n_ = int(re.sub(r"\D", "", Path(jp).stem) or 0)
                    r = row_of.get((v, n_))
                    if r is None:
                        continue
                    d = jload(jp)
                    if not d:
                        continue
                    names = {str(nm).lower() for nm, sc in
                             zip(d.get("detection_class_entities", []), d.get("detection_scores", []))
                             if float(sc) >= CFG["OBJ_MIN_SCORE"]}
                    if names:
                        OBJ_SETS[r] = names
                        out.write(json.dumps([r, sorted(names)]) + "\n")
                if vi_ % 100 == 0:
                    print(f"  objects {vi_}/{len(VIDEOS)}  {time.time()-t0:.0f}s", flush=True)
print("keyframes có object:", sum(1 for s in OBJ_SETS if s))

## 5. Index truy hồi: BM25 + dense (keyframe & video level)

In [ ]:
def norm_txt(s):
    s = unicodedata.normalize("NFC", (s or "").lower())
    return re.sub(r"[^\w\s]", " ", s)

KF_DOC    = [f"{CAP[i]} || {OCRT[i]} || {ASR_EN[i]} || {ASR_VI[i]}" for i in range(N)]
DENSE_DOC = [f"{CAP[i]} | text on screen: {OCRT[i]}" for i in range(N)]
VID_DOC   = {v: " ".join([SUMM.get(v, ""),
                          TRANS.get(v, {}).get("en", "")[:8000],
                          TRANS.get(v, {}).get("vi", "")[:8000],
                          MEDIA.get(v, "")]) for v in VIDEOS}
print(KF_DOC[1][:300])

In [ ]:
import bm25s

def build_bm25(docs, cache):
    if os.path.isdir(cache):
        return bm25s.BM25.load(cache, mmap=False)
    r = bm25s.BM25()
    r.index(bm25s.tokenize([norm_txt(d) for d in docs], stopwords=None, stemmer=None,
                           show_progress=False))
    r.save(cache)
    return r

BM_KF    = build_bm25(KF_DOC, os.path.join(CFG["CACHE_DIR"], "bm25_kf"))
VID_LIST = VIDEOS
BM_VID   = build_bm25([VID_DOC[v] for v in VID_LIST], os.path.join(CFG["CACHE_DIR"], "bm25_vid"))

def bm25_scores(retriever, n_docs, query, k=3000):
    q = norm_txt(query).strip()
    out = np.zeros(n_docs, np.float32)
    if not q:
        return out
    toks = bm25s.tokenize([q], stopwords=None, stemmer=None, show_progress=False)
    docs, sc = retriever.retrieve(toks, k=min(k, n_docs), show_progress=False)
    out[docs[0]] = sc[0]
    return out

print("BM25 ready.")

In [ ]:
from sentence_transformers import SentenceTransformer

# ---- CLIP text encoder: cùng không gian với feature BTC cấp ----
CLIP_TXT = SentenceTransformer(CFG["CLIP_MODEL"], device=DEV)

def clip_scores(prompts):
    """max cosine trên tập prompt -> (N,)"""
    prompts = [p for p in prompts if p and p.strip()]
    if not prompts:
        return np.zeros(N, np.float32)
    e = CLIP_TXT.encode(prompts, convert_to_numpy=True, normalize_embeddings=True,
                        batch_size=64, show_progress_bar=False)
    q = torch.from_numpy(e.astype(np.float16)).to(DEV)
    return (CLIP_T @ q.T).float().max(dim=1).values.cpu().numpy()

_t = clip_scores(["a man in a red shirt speaking at an outdoor press conference"])
print("clip_scores ok:", _t.shape, float(_t.max()))

In [ ]:
# ---- Dense encoder trên keyframe docs — dùng cả 2 GPU, có cache ----
DENSE_CACHE = os.path.join(CFG["CACHE_DIR"], "dense_kf.npy")
DENSE_T = DENSE_ENC = None

if CFG["USE_DENSE"]:
    DENSE_ENC = SentenceTransformer(CFG["DENSE_MODEL"], device=DEV)
    if os.path.exists(DENSE_CACHE):
        emb = np.load(DENSE_CACHE, mmap_mode="r")
    else:
        texts = ["passage: " + (d if d.strip(" |") else "empty")[:900] for d in DENSE_DOC]
        t0 = time.time()
        if NGPU > 1:
            pool = DENSE_ENC.start_multi_process_pool([f"cuda:{i}" for i in range(NGPU)])
            emb = DENSE_ENC.encode_multi_process(texts, pool, batch_size=CFG["DENSE_BATCH"],
                                                 normalize_embeddings=True)
            DENSE_ENC.stop_multi_process_pool(pool)
        else:
            emb = DENSE_ENC.encode(texts, batch_size=CFG["DENSE_BATCH"], normalize_embeddings=True,
                                   convert_to_numpy=True, show_progress_bar=True)
        emb = np.asarray(emb, np.float16)
        np.save(DENSE_CACHE, emb)
        print(f"dense encode {time.time()-t0:.0f}s")
    DENSE_T = torch.from_numpy(np.ascontiguousarray(emb)).to(DEV)
    print("dense matrix:", tuple(DENSE_T.shape))

def dense_scores(queries):
    queries = [q for q in (queries or []) if q and q.strip()]
    if DENSE_T is None or not queries:
        return np.zeros(N, np.float32)
    q = DENSE_ENC.encode(["query: " + x for x in queries], normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=False).astype(np.float16)
    return (DENSE_T @ torch.from_numpy(q).to(DEV).T).float().max(dim=1).values.cpu().numpy()

In [ ]:
# ---- Dense video-level + video prior ----
VDENSE_CACHE = os.path.join(CFG["CACHE_DIR"], "dense_vid.npy")
VDENSE_T = None
if CFG["USE_DENSE"]:
    if os.path.exists(VDENSE_CACHE):
        vemb = np.load(VDENSE_CACHE)
    else:
        vtexts = ["passage: " + (VID_DOC[v][:1800] or "empty") for v in VID_LIST]
        vemb = DENSE_ENC.encode(vtexts, batch_size=32, normalize_embeddings=True,
                                convert_to_numpy=True, show_progress_bar=True).astype(np.float16)
        np.save(VDENSE_CACHE, vemb)
    VDENSE_T = torch.from_numpy(vemb).to(DEV)
    print("video dense:", tuple(VDENSE_T.shape))

VID2POS  = {v: i for i, v in enumerate(VID_LIST)}
ROW_VPOS = np.array([VID2POS[v] for v in VID_ARR], np.int64)

def video_prior(queries):
    """Điểm topical mỗi video, chuẩn hoá 0..1."""
    queries = [q for q in (queries or []) if q and q.strip()]
    s = np.zeros(len(VID_LIST), np.float32)
    if not queries:
        return s
    if VDENSE_T is not None:
        q = DENSE_ENC.encode(["query: " + x for x in queries], normalize_embeddings=True,
                             convert_to_numpy=True, show_progress_bar=False).astype(np.float16)
        s += (VDENSE_T @ torch.from_numpy(q).to(DEV).T).float().max(dim=1).values.cpu().numpy()
    b = bm25_scores(BM_VID, len(VID_LIST), " ".join(queries), k=len(VID_LIST))
    if b.max() > 0:
        s += 0.5 * (b / b.max())
    return (s - s.min()) / (s.max() - s.min() + 1e-8)

## 6. LLM / VLM API (Anthropic)

Key lấy từ Kaggle Secrets (`ANTHROPIC_API_KEY`) hoặc biến môi trường.
Nếu không có key → `LLM_ON = False`, pipeline tự fallback sang dùng trực tiếp text truy vấn.

In [ ]:
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
if not API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        API_KEY = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
    except Exception as e:
        print("Không lấy được secret:", e)

LLM_ON = bool(API_KEY) and CFG["USE_LLM"]
client = None
if LLM_ON:
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)
print("LLM_ON =", LLM_ON)

LLM_CACHE_PATH = os.path.join(CFG["CACHE_DIR"], "llm_cache.json")
LLM_CACHE = jload(LLM_CACHE_PATH) or {}

def llm(prompt, system=None, images=None, max_tokens=4000, model=None, cache=True):
    """Gọi Claude. images = list bytes JPEG/PNG. Trả về text. Có cache trên đĩa."""
    if not LLM_ON:
        return ""
    model = model or CFG["LLM_MODEL"]
    max_tokens = max(max_tokens, 8000)   # adaptive thinking cũng tiêu tokens từ max_tokens
    key = hashlib.sha256(json.dumps(
        [model, system, prompt, [hashlib.sha256(b).hexdigest() for b in (images or [])]],
        ensure_ascii=False).encode()).hexdigest()
    if cache and key in LLM_CACHE:
        return LLM_CACHE[key]

    content = []
    for b in (images or []):
        content.append({"type": "image", "source": {"type": "base64", "media_type": "image/jpeg",
                                                     "data": base64.b64encode(b).decode()}})
    content.append({"type": "text", "text": prompt})

    kw = dict(model=model, max_tokens=max_tokens,
              thinking={"type": "adaptive"},
              messages=[{"role": "user", "content": content}])
    if system:
        kw["system"] = system

    last = None
    for attempt in range(4):
        try:
            with client.messages.stream(**kw) as st:
                msg = st.get_final_message()
            if getattr(msg, "stop_reason", None) == "refusal":
                return ""
            txt = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
            if cache:
                LLM_CACHE[key] = txt
                io.open(LLM_CACHE_PATH, "w", encoding="utf-8").write(
                    json.dumps(LLM_CACHE, ensure_ascii=False))
            return txt
        except Exception as e:
            last = e
            time.sleep(3 * (attempt + 1))
    print("LLM lỗi:", last)
    return ""

import hashlib

def parse_json(txt, default=None):
    """Bóc JSON đầu tiên trong text (chịu được ```json fence và prose xung quanh)."""
    if not txt:
        return default
    txt = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
    for opener, closer in (("{", "}"), ("[", "]")):
        i = txt.find(opener)
        while i != -1:
            depth = 0
            for j in range(i, len(txt)):
                if txt[j] == opener:
                    depth += 1
                elif txt[j] == closer:
                    depth -= 1
                    if depth == 0:
                        try:
                            return json.loads(txt[i:j + 1])
                        except Exception:
                            break
            i = txt.find(opener, i + 1)
    return default

## 7. Query Understanding

In [ ]:
QUERY_FILES = sorted(glob.glob(os.path.join(P["query"], "query-*.txt")),
                     key=lambda p: (int(re.search(r"-(\d+)-", Path(p).name).group(1)), p))

def qtype(path):
    n = Path(path).stem
    return "trake" if n.endswith("trake") else ("qa" if n.endswith("qa") else "kis")

QUERIES = [dict(file=p, name=Path(p).stem, type=qtype(p),
                text=io.open(p, encoding="utf-8").read().strip()) for p in QUERY_FILES]
print(f"{len(QUERIES)} truy vấn:",
      Counter(q["type"] for q in QUERIES))
for q in QUERIES:
    print(f"  {q['name']:22s} [{q['type']}] {q['text'][:70]}...")

In [ ]:
QU_SYS = """Bạn là chuyên gia truy hồi video cho cuộc thi AI Challenge (HCMC AIC).
Kho dữ liệu: ~873 video tiếng Việt từ YouTube (bản tin HTV "60 Giây", phóng sự, du lịch,
ẩm thực/nấu ăn, thể thao, khoa học, lễ hội...). Mỗi video đã có sẵn:
- caption tiếng ANH cho từng keyframe (Florence-2, mô tả những gì nhìn thấy)
- OCR tiếng VIỆT chữ trên màn hình (tên địa danh, banner, biển hiệu, phụ đề, logo)
- ASR tiếng Việt + bản dịch tiếng Anh của lời thoại
- CLIP ViT-B/32 image features
- object detection (nhãn OpenImages tiếng Anh, ví dụ: Person, Bicycle, Food, Tree, Dog)

Nhiệm vụ: chuyển truy vấn tiếng Việt của ban giám khảo thành các tín hiệu truy hồi.
Chỉ trả về JSON, không giải thích."""

QU_TMPL = """Truy vấn (tiếng Việt):
\"\"\"{text}\"\"\"

Loại truy vấn: {qtype}

Trả về JSON đúng schema sau:
{{
  "topic_en": "1-2 câu tiếng Anh mô tả chủ đề/bối cảnh tổng thể của video cần tìm (để khớp với summary & transcript)",
  "topic_vi": "1-2 câu tiếng Việt tương ứng (để khớp transcript tiếng Việt)",
  "clip_prompts": ["3-6 câu tiếng Anh NGẮN (<20 từ), mô tả CẢNH NHÌN THẤY ĐƯỢC, viết theo văn phong caption ảnh. Mỗi câu là một cách diễn đạt khác nhau của cùng cảnh đó. Chỉ tả thứ nhìn thấy: người/trang phục/màu sắc/vật thể/bối cảnh/góc quay."],
  "keywords_vi": ["8-15 từ khoá tiếng Việt có thể xuất hiện trong lời thoại hoặc chữ trên màn hình"],
  "keywords_en": ["8-15 từ khoá tiếng Anh có thể xuất hiện trong caption"],
  "ocr_terms": ["0-8 chuỗi chữ CỤ THỂ có khả năng cao hiện trên màn hình (tên riêng, địa danh, tên chương trình, tên tổ chức). Bỏ trống nếu không đoán được"],
  "objects": ["0-8 nhãn OpenImages tiếng Anh liên quan, viết thường, ví dụ: person, bicycle, food, dog"],
  "question_en": "CHỈ với loại qa: câu hỏi dịch sang tiếng Anh, ngược lại để \\"\\"",
  "answer_where": "CHỈ với loại qa: gợi ý đáp án nằm ở đâu — một trong \\"ocr\\", \\"speech\\", \\"visual\\"",
  "events": [
    {{"id": 1,
      "prompt_en": "câu tiếng Anh ngắn kiểu caption cho khoảnh khắc này",
      "keywords_en": ["..."]}}
  ]
}}

Ghi chú:
- "events" CHỈ dùng cho loại trake: liệt kê đúng {n_events} event theo thứ tự thời gian E1..E{n_events}.
  Với loại khác để "events": [].
- clip_prompts rất quan trọng: chúng được so khớp trực tiếp với CLIP image features nên phải
  là mô tả thị giác cụ thể, KHÔNG phải suy luận hay câu hỏi.
- Nếu truy vấn dựa trên tri thức ngoài (ví dụ "loài cá trong phim của Steven Spielberg 1975"),
  hãy GIẢI luôn tri thức đó và đưa đáp án vào clip_prompts/keywords (ví dụ "great white shark").
"""

def n_events_of(text):
    ids = re.findall(r"^\s*E\s*(\d+)\s*[:.]", text, flags=re.M | re.I)
    return max(len(ids), len(set(ids))) or 4

def fallback_parse(q):
    t = q["text"]
    return dict(topic_en=t[:400], topic_vi=t[:400], clip_prompts=[t[:250]],
                keywords_vi=[w for w in vi_words(t)][:20], keywords_en=[],
                ocr_terms=[], objects=[], question_en="", answer_where="visual",
                events=[dict(id=i + 1, prompt_en=l, keywords_en=[])
                        for i, l in enumerate(re.findall(r"^\s*E\s*\d+\s*[:.]\s*(.+)$", t,
                                                         flags=re.M | re.I))])

def vi_words(s):
    return [w for w in norm_txt(s).split() if len(w) > 2]

def understand(q):
    ne = n_events_of(q["text"]) if q["type"] == "trake" else 0
    if not LLM_ON:
        return fallback_parse(q)
    out = parse_json(llm(QU_TMPL.format(text=q["text"], qtype=q["type"], n_events=ne),
                         system=QU_SYS, max_tokens=3000))
    if not isinstance(out, dict):
        return fallback_parse(q)
    fb = fallback_parse(q)
    for k, v in fb.items():
        out.setdefault(k, v)
    for k in ("clip_prompts", "keywords_vi", "keywords_en", "ocr_terms", "objects", "events"):
        if not isinstance(out.get(k), list):
            out[k] = []
    if q["type"] == "trake":
        ev = out["events"] or fb["events"]
        if len(ev) != ne:                      # số event phải khớp đề bài
            ev = (ev + fb["events"])[:ne] if len(ev) < ne else ev[:ne]
        while len(ev) < ne:
            ev.append(dict(id=len(ev) + 1, prompt_en=out["topic_en"], keywords_en=[]))
        out["events"] = [dict(id=i + 1,
                              prompt_en=(e.get("prompt_en") or out["topic_en"]),
                              keywords_en=e.get("keywords_en") or [])
                         for i, e in enumerate(ev)]
    else:
        out["events"] = []
    return out

PARSED = {}
for q in QUERIES:
    PARSED[q["name"]] = understand(q)
    p = PARSED[q["name"]]
    print(f"\n### {q['name']} [{q['type']}]  events={len(p['events'])}")
    print("  topic:", (p.get('topic_en') or '')[:110])
    for cp in p["clip_prompts"][:4]:
        print("   clip:", cp)
    if p.get("ocr_terms"):
        print("   ocr :", p["ocr_terms"])
    for e in p["events"]:
        print(f"   E{e['id']}:", e["prompt_en"][:100])

## 8. Fusion — gộp điểm nhiều tín hiệu

In [ ]:
def mm(x):
    lo, hi = float(np.min(x)), float(np.max(x))
    return (x - lo) / (hi - lo + 1e-8)

def ocr_bonus(rows, terms):
    """Khớp chuỗi con của các cụm OCR dự đoán vào text OCR thật của keyframe."""
    if not terms:
        return np.zeros(len(rows), np.float32)
    terms_n = [norm_txt(t).strip() for t in terms if len(norm_txt(t).strip()) >= 3]
    if not terms_n:
        return np.zeros(len(rows), np.float32)
    out = np.zeros(len(rows), np.float32)
    for i, r in enumerate(rows):
        s = norm_txt(OCRT[r])
        if not s:
            continue
        hit = sum(1 for t in terms_n if t in s)
        # khớp một phần theo token cho cụm nhiều từ
        part = 0.0
        for t in terms_n:
            tk = t.split()
            if len(tk) > 1:
                part += sum(1 for w in tk if w in s) / len(tk)
        out[i] = hit + 0.4 * part
    return mm(out) if out.max() > 0 else out

def obj_bonus(rows, objs):
    if not objs:
        return np.zeros(len(rows), np.float32)
    want = {str(o).lower() for o in objs}
    return np.array([len(want & OBJ_SETS[r]) / len(want) for r in rows], np.float32)

def retrieve(parsed, extra_clip=None, extra_kw=None, topn=4000):
    """Trả về (rows, scores) đã sắp giảm dần theo điểm gộp."""
    clip_prompts = list(parsed["clip_prompts"]) + list(extra_clip or [])
    if parsed.get("topic_en"):
        clip_prompts.append(parsed["topic_en"][:180])

    dense_q = [x for x in ([parsed.get("topic_en"), parsed.get("topic_vi")]
                           + clip_prompts
                           + [" ".join(parsed.get("keywords_en") or [])[:300],
                              " ".join(parsed.get("keywords_vi") or [])[:300]]) if x]
    bm_q = " ".join((parsed.get("keywords_vi") or []) + (parsed.get("keywords_en") or [])
                    + (parsed.get("ocr_terms") or []) + list(extra_kw or []))

    s_clip  = clip_scores(clip_prompts)
    s_dense = dense_scores(dense_q)
    s_bm    = bm25_scores(BM_KF, N, bm_q, k=CFG["POOL_PER_LIST"] * 2)
    vp      = video_prior([x for x in (parsed.get("topic_en"), parsed.get("topic_vi")) if x])

    K = CFG["POOL_PER_LIST"]
    pool = set()
    for arr in (s_clip, s_dense, s_bm):
        if arr.max() > 0:
            pool.update(np.argpartition(-arr, min(K, N - 1))[:K].tolist())
    rows = np.array(sorted(pool), np.int64)

    tot = (CFG["W_CLIP"]  * mm(s_clip[rows])
           + CFG["W_DENSE"] * mm(s_dense[rows])
           + CFG["W_BM25"]  * mm(s_bm[rows])
           + CFG["W_OCR"]   * ocr_bonus(rows, parsed.get("ocr_terms"))
           + CFG["W_OBJ"]   * obj_bonus(rows, parsed.get("objects"))
           + CFG["W_VPRIOR"] * vp[ROW_VPOS[rows]])

    order = np.argsort(-tot)[:topn]
    return rows[order], tot[order]

def evidence(r, maxlen=430):
    v = VID_ARR[r]
    parts = [f"video={v} frame={int(FRAME_IDX[r])} t={PTS[r]:.1f}s"]
    if CAP[r]:  parts.append("CAPTION: " + CAP[r][:220])
    if OCRT[r]: parts.append("OCR: " + OCRT[r][:110])
    if ASR_VI[r]: parts.append("SPEECH: " + ASR_VI[r][:150])
    return " | ".join(parts)[:maxlen]

## 9. Rerank bằng LLM (text evidence) và VLM (ảnh keyframe)

In [ ]:
RR_SYS = ("Bạn chấm mức khớp giữa truy vấn tiếng Việt và các keyframe video. "
          "Bằng chứng gồm caption tiếng Anh (Florence-2, có thể sai chi tiết nhỏ), "
          "OCR tiếng Việt và lời thoại. Chỉ trả về JSON.")

def llm_rerank(query_text, rows, scores, topk=None):
    topk = topk or CFG["LLM_RERANK_TOPK"]
    if not LLM_ON or not CFG["USE_LLM_RERANK"] or len(rows) == 0:
        return rows, scores
    k = min(topk, len(rows))
    cand = "\n".join(f"[{i}] {evidence(int(rows[i]))}" for i in range(k))
    txt = llm(
        f"""Truy vấn: \"\"\"{query_text}\"\"\"

Danh sách keyframe ứng viên:
{cand}

Cho điểm 0-100 mức khớp của từng ứng viên với truy vấn (0 = không liên quan).
Ưu tiên khớp các chi tiết thị giác cụ thể (màu áo, số người, vật thể, bối cảnh) và
khớp chủ đề. Trả về JSON: {{"scores": {{"0": 87, "1": 12, ...}}}} — đủ mọi chỉ số 0..{k-1}.""",
        system=RR_SYS, max_tokens=3000)
    d = parse_json(txt, {}) or {}
    sc = d.get("scores", d) if isinstance(d, dict) else {}
    boost = np.zeros(len(rows), np.float32)
    got = 0
    for i in range(k):
        try:
            boost[i] = float(sc[str(i)]) / 100.0
            got += 1
        except Exception:
            pass
    if got < k * 0.5:
        return rows, scores
    new = scores.copy()
    new[k:] = mm(scores[k:]) if len(scores) > k else new[k:]        # nhóm chưa rerank: 0..1
    new[:k] = 2.0 + boost[:k] * 2.0 + 0.2 * mm(scores[:k])          # nhóm đã rerank: luôn ở trên
    o = np.argsort(-new)
    return rows[o], new[o]

def keyframe_path(r):
    v, n_ = VID_ARR[r], int(KF_N[r])
    for base in P["keyframes"]:
        for pat in (f"{n_:03d}.jpg", f"{n_:04d}.jpg", f"{n_}.jpg"):
            p = os.path.join(base, v, pat)
            if os.path.exists(p):
                return p
    hits = glob.glob(os.path.join("/kaggle/input", "**", v, f"*{n_:03d}.jpg"), recursive=True)
    return hits[0] if hits else None

def vlm_rerank(query_text, rows, scores, topk=None):
    topk = topk or CFG["VLM_RERANK_TOPK"]
    if not LLM_ON or not CFG["USE_VLM_RERANK"] or not P["keyframes"] or len(rows) == 0:
        return rows, scores
    k, imgs, keep = min(topk, len(rows)), [], []
    for i in range(k):
        p = keyframe_path(int(rows[i]))
        if p:
            imgs.append(open(p, "rb").read())
            keep.append(i)
    if not imgs:
        return rows, scores
    txt = llm(f"""Truy vấn: \"\"\"{query_text}\"\"\"

Có {len(imgs)} ảnh keyframe theo thứ tự, đánh số 0..{len(imgs)-1}.
Cho điểm 0-100 mức khớp của TỪNG ảnh với truy vấn, xét đúng các chi tiết thị giác
(số người, màu trang phục, vật thể, bối cảnh, góc quay).
Trả về JSON: {{"scores": {{"0": 91, ...}}}}""",
              system="Bạn chấm mức khớp giữa ảnh keyframe và truy vấn tiếng Việt. Chỉ trả JSON.",
              images=imgs, max_tokens=2000, model=CFG["VLM_MODEL"])
    d = parse_json(txt, {}) or {}
    sc = d.get("scores", d) if isinstance(d, dict) else {}
    new = scores.copy()
    hit = 0
    for j, i in enumerate(keep):
        try:
            new[i] = 6.0 + float(sc[str(j)]) / 100.0 * 3.0   # đẩy hẳn lên trên mọi nhóm trước
            hit += 1
        except Exception:
            pass
    if hit == 0:
        return rows, scores
    o = np.argsort(-new)
    return rows[o], new[o]

## 10. Sinh dòng nộp — hedging frame-spread

Đáp án đúng chỉ tính khi `frame_id ∈ [s, e]` mà khoảng này rất hẹp, còn keyframe lại thưa
(trung bình vài chục → vài trăm frame một keyframe). Vì vậy mỗi ứng viên tốt được trải thêm
vài frame trong cùng shot (giữa keyframe trước và sau) ở các hạng phía sau.

In [ ]:
def shot_bounds(r):
    """(prev_frame, frame, next_frame) trong cùng video."""
    v = VID_ARR[r]
    rws = VID2ROWS[v]
    pos = int(np.searchsorted(rws, r))
    f  = int(FRAME_IDX[r])
    pf = int(FRAME_IDX[rws[pos - 1]]) if pos > 0 else max(0, f - 90)
    nf = int(FRAME_IDX[rws[pos + 1]]) if pos + 1 < len(rws) else f + 90
    return pf, f, nf

def spread_frames(r, k=6):
    """Frame đầu tiên = chính keyframe; các frame sau trải trong shot."""
    pf, f, nf = shot_bounds(r)
    fwd, bwd = max(1, nf - f), max(1, f - pf)
    cands = [f,
             f + fwd // 3, f - bwd // 3,
             f + 2 * fwd // 3, f - 2 * bwd // 3,
             f + fwd // 6, f - bwd // 6,
             f + int(0.85 * fwd), f - int(0.85 * bwd)]
    out, seen = [], set()
    for c in cands:
        c = max(0, int(c))
        if c not in seen:
            seen.add(c)
            out.append(c)
        if len(out) >= k:
            break
    return out

def build_rows(rows, n_rows=None, extra=None):
    """Xếp: primary top-20 -> spread của top-10 -> primary còn lại -> spread thêm.
    extra(video, frame) -> tuple thêm vào cuối mỗi dòng (dùng cho Q&A answer)."""
    n_rows = n_rows or CFG["N_ROWS"]
    out, seen = [], set()

    def push(r, frame):
        v = VID_ARR[r]
        key = (v, int(frame))
        if key in seen:
            return
        seen.add(key)
        rec = [v, int(frame)]
        if extra:
            rec += list(extra(v, int(frame), r))
        out.append(rec)

    rows = [int(r) for r in rows]
    for r in rows[:20]:                                   # hạng 1-20: đa dạng ứng viên
        push(r, spread_frames(r, 1)[0])
        if len(out) >= n_rows: return out[:n_rows]
    for j in range(1, 4):                                 # hedge trong shot của top-10
        for r in rows[:10]:
            sp = spread_frames(r, j + 1)
            if len(sp) > j:
                push(r, sp[j])
            if len(out) >= n_rows: return out[:n_rows]
    for r in rows[20:]:
        push(r, spread_frames(r, 1)[0])
        if len(out) >= n_rows: return out[:n_rows]
    for j in range(1, 6):
        for r in rows[:60]:
            sp = spread_frames(r, j + 1)
            if len(sp) > j:
                push(r, sp[j])
            if len(out) >= n_rows: return out[:n_rows]
    return out[:n_rows]

## 11. Pipeline Textual KIS

In [ ]:
def run_kis(q):
    p = PARSED[q["name"]]
    rows, sc = retrieve(p)
    rows, sc = llm_rerank(q["text"], rows, sc)
    rows, sc = vlm_rerank(q["text"], rows, sc)
    print(f"  top5: " + " | ".join(f"{VID_ARR[r]}@{int(FRAME_IDX[r])}" for r in rows[:5]))
    return build_rows(rows), rows, sc

## 12. Pipeline Q&A

Truy hồi cảnh → gom bằng chứng (OCR quanh keyframe + lời thoại + caption) → LLM trả lời.
Notebook lấy đáp án chính cho các hạng đầu và giữ 1-2 đáp án dự phòng ở hạng sau
(vì `R@k` chỉ cần **một** dòng đúng trong top-k).

In [ ]:
QA_SYS = ("Bạn trả lời câu hỏi về nội dung video tiếng Việt, chỉ dựa trên bằng chứng được cung cấp "
          "(OCR chữ trên màn hình, lời thoại ASR, caption ảnh). Đáp án phải NGẮN GỌN — "
          "chỉ đúng thông tin được hỏi, tối đa 100 ký tự, không giải thích. Chỉ trả về JSON.")

def qa_context(rows, k_scene=8, k_ctx=6):
    """Gom bằng chứng quanh top ứng viên: keyframe lân cận ±k_ctx trong cùng video."""
    blocks, used_v = [], []
    for r in rows:
        v = VID_ARR[r]
        if sum(1 for x in used_v if x == v) >= 3:
            continue
        used_v.append(v)
        rws = VID2ROWS[v]
        pos = int(np.searchsorted(rws, r))
        near = rws[max(0, pos - k_ctx): pos + k_ctx + 1]
        ocr = " / ".join(sorted({OCRT[x] for x in near if OCRT[x]}))[:700]
        caps = " / ".join(dict.fromkeys(CAP[x][:110] for x in near if CAP[x]))[:700]
        speech = " ".join(dict.fromkeys(ASR_VI[x] for x in near if ASR_VI[x]))[:900]
        blocks.append(f"""--- ỨNG VIÊN {len(blocks)} : video={v}, frame={int(FRAME_IDX[r])}, t={PTS[r]:.1f}s
SUMMARY VIDEO: {(SUMM.get(v, '') or '')[:400]}
OCR quanh cảnh: {ocr}
LỜI THOẠI quanh cảnh: {speech}
CAPTION quanh cảnh: {caps}""")
        if len(blocks) >= k_scene:
            break
    return "\n".join(blocks)

def run_qa(q):
    p = PARSED[q["name"]]
    rows, sc = retrieve(p)
    rows, sc = llm_rerank(q["text"], rows, sc)
    rows, sc = vlm_rerank(q["text"], rows, sc)

    ans_main, ans_alts, best_i = "", [], 0
    if LLM_ON:
        ctx = qa_context([int(r) for r in rows[:80]])
        txt = llm(f"""Truy vấn / câu hỏi (tiếng Việt):
\"\"\"{q['text']}\"\"\"

Bằng chứng từ các ứng viên truy hồi được:
{ctx}

Hãy chọn ứng viên đúng và trả lời câu hỏi.
Trả về JSON:
{{"best_candidate": <số thứ tự ứng viên>,
  "answer": "đáp án ngắn gọn, tối đa 100 ký tự",
  "alt_answers": ["1-2 đáp án dự phòng khác cách diễn đạt hoặc khả năng thứ hai"],
  "confidence": 0-100,
  "reason": "một câu ngắn"}}

Lưu ý: nếu đáp án là tên riêng/địa danh thì viết đúng chính tả tiếng Việt có dấu.
Nếu là số lượng thì trả về chữ số.""", system=QA_SYS, max_tokens=2500)
        d = parse_json(txt, {}) or {}
        ans_main = str(d.get("answer", ""))[:100].strip()
        ans_alts = [str(a)[:100].strip() for a in (d.get("alt_answers") or []) if str(a).strip()][:2]
        print("  answer:", repr(ans_main), "| alts:", ans_alts,
              "| conf:", d.get("confidence"), "|", str(d.get("reason", ""))[:90])

        # đưa ứng viên mà LLM chọn lên hạng 1
        try:
            bi = int(d["best_candidate"])
            seen_v, order_map = [], []
            for i, r in enumerate(rows[:80]):
                v = VID_ARR[int(r)]
                if sum(1 for x in seen_v if x == v) >= 3:
                    continue
                seen_v.append(v)
                order_map.append(i)
            if 0 <= bi < len(order_map):
                best_i = order_map[bi]
                rows = np.concatenate([[rows[best_i]], np.delete(rows, best_i)])
        except Exception:
            pass
    if not ans_main:
        ans_main = "không xác định"

    answers = [ans_main] + ans_alts
    def extra(v, frame, r):
        return (ans_main,)
    recs = build_rows(rows, n_rows=CFG["N_ROWS"], extra=extra)

    # thay đáp án dự phòng vào một phần hạng sau (giữ nguyên video/frame tốt nhất)
    if ans_alts and len(recs) >= 12:
        slots = list(range(6, min(len(recs), 6 + 4 * len(ans_alts))))
        for j, s in enumerate(slots):
            recs[s] = [recs[s][0], recs[s][1], ans_alts[j % len(ans_alts)]]
    return recs, rows, sc

## 13. Pipeline TRAKE

1. Tính điểm riêng cho từng event trên toàn bộ keyframe.
2. Với mỗi video ứng viên, **DP căn chỉnh đơn điệu** (frame của E1 < E2 < ... < EN) → điểm video.
3. Chọn video tốt nhất, *(tuỳ chọn)* **fine-align ở mức frame** bằng cách giải mã video gốc
   và so khớp CLIP ảnh-frame với prompt của từng event.
4. Sinh 100 dòng: quét offset toàn cục + jitter theo từng event + các video dự phòng.

In [ ]:
def event_score_matrix(parsed):
    """(E, N) điểm từng event trên toàn bộ keyframe."""
    S = []
    for e in parsed["events"]:
        pr = [e["prompt_en"]]
        s = (CFG["W_CLIP"] * mm(clip_scores(pr))
             + CFG["W_DENSE"] * mm(dense_scores([e["prompt_en"]]))
             + 0.3 * mm(bm25_scores(BM_KF, N, " ".join(e.get("keywords_en") or []) or e["prompt_en"],
                                    k=CFG["POOL_PER_LIST"])))
        S.append(s.astype(np.float32))
    return np.vstack(S)

def dp_align(S_v):
    """S_v: (E, M) điểm trên M keyframe của một video (theo thứ tự thời gian).
    Trả về (tổng điểm tốt nhất, [chỉ số keyframe cho từng event]) với ràng buộc tăng dần."""
    E, M = S_v.shape
    if M < E:
        idx = list(range(M)) + [M - 1] * (E - M)
        return float(S_v[np.arange(E), idx].sum()), idx
    NEG = -1e9
    dp = np.full((E, M), NEG, np.float32)
    bk = np.zeros((E, M), np.int32)
    dp[0] = S_v[0]
    for e in range(1, E):
        best, arg = NEG, 0
        for j in range(M):
            if dp[e - 1, j - 1] > best and j >= 1:
                best, arg = dp[e - 1, j - 1], j - 1
            if j >= 1:
                dp[e, j] = best + S_v[e, j]
                bk[e, j] = arg
    j = int(np.argmax(dp[E - 1]))
    path = [j]
    for e in range(E - 1, 0, -1):
        j = int(bk[e, j])
        path.append(j)
    path.reverse()
    return float(dp[E - 1, path[-1]]), path

def trake_candidates(parsed, n_video=12):
    S = event_score_matrix(parsed)                       # (E, N)
    E = S.shape[0]
    vp = video_prior([x for x in (parsed.get("topic_en"), parsed.get("topic_vi")) if x])

    # lọc sơ bộ video theo max điểm event + prior
    vscore = np.zeros(len(VID_LIST), np.float32)
    for e in range(E):
        top = np.argpartition(-S[e], 4000)[:4000]
        agg = np.zeros(len(VID_LIST), np.float32)
        np.maximum.at(agg, ROW_VPOS[top], S[e][top])
        vscore += mm(agg)
    vscore = mm(vscore) + CFG["W_VPRIOR"] * vp
    shortlist = [VID_LIST[i] for i in np.argsort(-vscore)[:250]]

    out = []
    for v in shortlist:
        rws = VID2ROWS[v]
        if len(rws) < 2:
            continue
        S_v = S[:, rws]
        tot, path = dp_align(S_v)
        out.append(dict(video=v,
                        score=tot / E + CFG["W_VPRIOR"] * float(vp[VID2POS[v]]),
                        rows=[int(rws[j]) for j in path]))
    out.sort(key=lambda d: -d["score"])
    return out[:n_video], S

In [ ]:
# ---- Fine alignment ở mức frame trên video gốc (tuỳ chọn nhưng rất nên bật) ----
def find_video_file(v):
    for base in P["videos"]:
        for pat in (os.path.join(base, "video", f"{v}.mp4"), os.path.join(base, f"{v}.mp4")):
            if os.path.exists(pat):
                return pat
    hits = glob.glob(f"/kaggle/input/**/{v}.mp4", recursive=True)
    return hits[0] if hits else None

def fine_align(video_id, pred_frames, prompts):
    """Quét frame quanh mỗi frame dự đoán, chọn frame khớp CLIP nhất với prompt của event."""
    if not CFG["USE_VIDEO_FINE_ALIGN"]:
        return pred_frames, None
    try:
        import cv2
        from PIL import Image
    except Exception as e:
        print("  fine-align bỏ qua (thiếu cv2/PIL):", e)
        return pred_frames, None
    path = find_video_file(video_id)
    if not path:
        print("  fine-align bỏ qua: không thấy file video", video_id)
        return pred_frames, None

    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    nfr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    win = int(CFG["FINE_WIN_SEC"] * fps)
    txt_emb = CLIP_TXT.encode(prompts, convert_to_numpy=True, normalize_embeddings=True,
                              show_progress_bar=False)

    refined, curves = [], []
    lo_bound = 0
    for e, (f0, pr) in enumerate(zip(pred_frames, prompts)):
        lo = max(lo_bound, int(f0) - win)
        hi = min(nfr - 1 if nfr else int(f0) + win, int(f0) + win)
        if hi <= lo:
            refined.append(int(f0)); curves.append(None); continue
        idxs = list(range(lo, hi + 1, CFG["FINE_STRIDE"]))
        imgs, keep = [], []
        cap.set(cv2.CAP_PROP_POS_FRAMES, lo)
        cur = lo
        want = set(idxs)
        while cur <= hi:
            ok, frame = cap.read()
            if not ok:
                break
            if cur in want:
                imgs.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
                keep.append(cur)
            cur += 1
        if not imgs:
            refined.append(int(f0)); curves.append(None); continue
        iemb = CLIP_TXT.encode(imgs, convert_to_numpy=True, normalize_embeddings=True,
                               batch_size=64, show_progress_bar=False)
        s = iemb @ txt_emb[e]
        best = int(keep[int(np.argmax(s))])
        refined.append(best)
        curves.append((keep, s))
        lo_bound = best + 1          # giữ thứ tự thời gian giữa các event
    cap.release()
    print(f"  fine-align {video_id}: {pred_frames} -> {refined}")
    return refined, curves

In [ ]:
def trake_rows(cands, S, parsed, n_rows=None):
    """Sinh tối đa 100 dòng: quét offset toàn cục + jitter từng event + video dự phòng."""
    n_rows = n_rows or CFG["N_ROWS"]
    E = len(parsed["events"])
    prompts = [e["prompt_en"] for e in parsed["events"]]
    rows, seen = [], set()

    def push(v, frames):
        frames = [max(0, int(f)) for f in frames]
        for i in range(1, len(frames)):                  # bắt buộc tăng dần theo thời gian
            if frames[i] <= frames[i - 1]:
                frames[i] = frames[i - 1] + 1
        key = (v, tuple(frames))
        if key in seen or len(frames) != E:
            return
        seen.add(key)
        rows.append([v] + frames)

    top = cands[0]
    base_frames = [int(FRAME_IDX[r]) for r in top["rows"]]
    fine, curves = fine_align(top["video"], base_frames, prompts)
    push(top["video"], fine)

    fps = float(FPS_ARR[top["rows"][0]]) or 30.0

    # (a) đỉnh phụ từ đường cong CLIP fine-align (nếu có) — sát đáp án nhất
    if curves and any(c is not None for c in curves):
        for rank in range(1, 4):
            fr = []
            for e in range(E):
                c = curves[e] if e < len(curves) else None
                if c is None:
                    fr.append(fine[e]); continue
                keep, s = c
                o = np.argsort(-s)
                fr.append(int(keep[o[min(rank, len(o) - 1)]]))
            push(top["video"], fr)

    # (b) quét offset toàn cục (sai số hệ thống)
    step = max(2, int(0.10 * fps))
    for k in range(1, 26):
        d = ((k + 1) // 2) * step * (1 if k % 2 else -1)
        push(top["video"], [f + d for f in fine])

    # (c) jitter theo từng event
    js = [0, step, -step, 2 * step, -2 * step, 4 * step, -4 * step]
    rnd = random.Random(1234)
    while len(rows) < int(n_rows * 0.75):
        fr = [f + rnd.choice(js) for f in fine]
        push(top["video"], fr)
        if len(seen) > 4000:
            break

    # (d) video dự phòng (video sai => 0 điểm, nên vẫn nên phủ vài video khác)
    for c in cands[1:]:
        f2 = [int(FRAME_IDX[r]) for r in c["rows"]]
        push(c["video"], f2)
        for k in range(1, 7):
            d = ((k + 1) // 2) * step * (1 if k % 2 else -1)
            push(c["video"], [f + d for f in f2])
        if len(rows) >= n_rows:
            break
    return rows[:n_rows]

def run_trake(q):
    p = PARSED[q["name"]]
    cands, S = trake_candidates(p)
    print("  video ứng viên:", [(c["video"], round(c["score"], 3)) for c in cands[:5]])

    # LLM chọn lại video dựa trên bằng chứng chuỗi sự kiện
    if LLM_ON and CFG["USE_LLM_RERANK"] and len(cands) > 1:
        desc = []
        for i, c in enumerate(cands[:8]):
            ev = " ;; ".join(f"E{j+1}@{int(FRAME_IDX[r])}: {CAP[r][:110]}"
                             for j, r in enumerate(c["rows"]))
            desc.append(f"[{i}] video={c['video']}\n    SUMMARY: {(SUMM.get(c['video'],'') or '')[:300]}\n    {ev}")
        txt = llm(f"""Truy vấn TRAKE (tiếng Việt):
\"\"\"{q['text']}\"\"\"

Các video ứng viên cùng khoảnh khắc được căn chỉnh sơ bộ:
{chr(10).join(desc)}

Chọn video KHỚP NHẤT với toàn bộ chuỗi sự kiện.
Trả về JSON: {{"order": [danh sách chỉ số ứng viên, tốt nhất trước]}}""",
                  system=RR_SYS, max_tokens=1500)
        d = parse_json(txt, {}) or {}
        order = [i for i in (d.get("order") or []) if isinstance(i, int) and 0 <= i < len(cands)]
        if order:
            cands = [cands[i] for i in order] + [c for i, c in enumerate(cands) if i not in order]
            print("  -> LLM chọn:", cands[0]["video"])
    return trake_rows(cands, S, p), cands

## 14. Chạy toàn bộ truy vấn

In [ ]:
import csv

RESULTS = {}
for q in QUERIES:
    t0 = time.time()
    print(f"\n===== {q['name']} [{q['type']}] =====")
    print(" ", q["text"][:160].replace("\n", " "))
    try:
        if q["type"] == "kis":
            recs, rows, sc = run_kis(q)
        elif q["type"] == "qa":
            recs, rows, sc = run_qa(q)
        else:
            recs, cands = run_trake(q)
        RESULTS[q["name"]] = recs
        print(f"  -> {len(recs)} dòng | {time.time()-t0:.0f}s | dòng 1: {recs[0] if recs else None}")
    except Exception as e:
        import traceback; traceback.print_exc()
        RESULTS[q["name"]] = []
        print("  !! lỗi:", e)

## 15. Kiểm tra và xuất `submission.zip`

In [ ]:
def validate(name, qtype, recs, n_events=None):
    errs = []
    if not recs:
        errs.append("rỗng")
    if len(recs) > 100:
        errs.append(f"{len(recs)} dòng > 100")
    for i, r in enumerate(recs):
        if not re.fullmatch(r"L\d\d_V\d\d\d", str(r[0])):
            errs.append(f"dòng {i}: video_id sai '{r[0]}'")
        if qtype == "kis" and len(r) != 2:
            errs.append(f"dòng {i}: KIS phải có 2 cột, có {len(r)}")
        if qtype == "qa":
            if len(r) != 3:
                errs.append(f"dòng {i}: QA phải có 3 cột, có {len(r)}")
            elif len(str(r[2])) > 100:
                errs.append(f"dòng {i}: answer > 100 ký tự")
        if qtype == "trake" and n_events and len(r) != n_events + 1:
            errs.append(f"dòng {i}: TRAKE cần {n_events} frame, có {len(r)-1}")
        for f in r[1:] if qtype != "qa" else r[1:2]:
            if not (isinstance(f, (int, np.integer)) and int(f) >= 0):
                errs.append(f"dòng {i}: frame không phải số nguyên >= 0: {f!r}")
    if qtype == "trake" and len({tuple(r) for r in recs}) != len(recs):
        errs.append("có dòng trùng lặp")
    return errs

for f in glob.glob(os.path.join(CFG["OUT_DIR"], "*.csv")):
    os.remove(f)

ok = True
for q in QUERIES:
    recs = RESULTS.get(q["name"], [])
    ne = n_events_of(q["text"]) if q["type"] == "trake" else None
    errs = validate(q["name"], q["type"], recs, ne)
    out_path = os.path.join(CFG["OUT_DIR"], q["name"] + ".csv")
    with io.open(out_path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, quoting=csv.QUOTE_MINIMAL, lineterminator="\n")
        for r in recs:
            w.writerow([r[0]] + [int(x) if not isinstance(x, str) else x for x in r[1:]])
    tag = "OK " if not errs else "!! "
    if errs:
        ok = False
    print(f"{tag}{q['name']+'.csv':26s} {len(recs):3d} dòng" + ("  " + "; ".join(errs[:3]) if errs else ""))

if os.path.exists(CFG["ZIP_NAME"]):
    os.remove(CFG["ZIP_NAME"])
with zipfile.ZipFile(CFG["ZIP_NAME"], "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(CFG["OUT_DIR"], "*.csv"))):
        z.write(f, arcname=os.path.join("submission", os.path.basename(f)))   # PHẢI có thư mục submission/

print("\nzip:", CFG["ZIP_NAME"], os.path.getsize(CFG["ZIP_NAME"]), "bytes")
with zipfile.ZipFile(CFG["ZIP_NAME"]) as z:
    print("\n".join(z.namelist()))
print("\nTất cả file hợp lệ." if ok else "\nCÒN LỖI — xem cảnh báo phía trên.")

In [ ]:
# Xem nhanh 5 dòng đầu của mỗi file để mắt người kiểm tra lần cuối
for f in sorted(glob.glob(os.path.join(CFG["OUT_DIR"], "*.csv"))):
    print("=" * 70)
    print(os.path.basename(f))
    print(io.open(f, encoding="utf-8").read().split("\n")[:5])

## 16. Ghi chú vận hành

**Thứ tự chạy lần đầu (~40-60 phút, phần lớn là encode dense + index object, đều có cache):**
1. Chạy hết cell → sinh `cache/` trong `/kaggle/working`.
2. Nên **Save Version** để tái sử dụng cache: các lần sau chỉ mất vài phút cho phần truy vấn.

**Điều chỉnh khi kết quả chưa tốt**

| Hiện tượng | Cách xử lý |
|---|---|
| Đúng chủ đề nhưng sai video | tăng `W_VPRIOR`, thêm `ocr_terms` cụ thể (tên chương trình, địa danh) |
| Đúng video nhưng sai frame | bật `USE_VLM_RERANK`, giảm `POOL_PER_LIST`, tăng `W_CLIP` |
| Truy vấn cần tri thức ngoài (Spielberg 1975 → cá mập trắng; Lausanne → EPFL) | LLM đã tự giải; kiểm tra `PARSED[...]['clip_prompts']` in ra ở cell 7 |
| TRAKE sai frame | **bật `USE_VIDEO_FINE_ALIGN`** và add dataset `Videos_L*` — keyframe thưa nên gần như không thể trúng khoảng <10 frame nếu không quét ở mức frame |
| Q&A đúng cảnh, sai đáp án | xem lại `answer_where`; nếu đáp án nằm ở chữ trên màn hình thì bật `USE_VLM_RERANK` để VLM đọc trực tiếp ảnh |

**Chọn bài nộp**: mỗi gói được nộp tối đa 3 lần, chỉ **lần cuối** tính điểm — nên nộp thử 1 lần
để xem Public LB (50% đáp án), rồi sửa cấu hình và nộp lần cuối.

**Kiểm tra bắt buộc trước khi nộp**: zip phải chứa thư mục `submission/`, file `.csv` thuần UTF-8,
không header, `video_id` không có `.mp4`, số frame của TRAKE khớp số event, answer Q&A ≤ 100 ký tự —
cell 15 đã kiểm tự động tất cả các mục này.